In [3]:
import os
import time
import re
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("Limpieza completada.")

# Conexion MongoDB
try:
    client = MongoClient('mongodb://database:27017/')
    db = client['proyecto_bigdata']
    coleccion = db['datos_turismo']
    print("MongoDB conectado!")
except Exception as e:
    print(f"Error MongoDB: {e}")

# Ciudades de Chile
ciudades = [
    ("Santiago", "Centro"),
    ("Valparaiso", "Centro"),
    ("Vina del Mar", "Centro"),
    ("Antofagasta", "Norte"),
    ("Iquique", "Norte"),
    ("Arica", "Norte"),
    ("Puerto Montt", "Sur"),
    ("Pucon", "Sur"),
    ("Puerto Varas", "Sur"),
]

# URLs de HotelsCombined
urls_hotelscombined = {
    "Santiago": "https://www.hotelscombined.com/Place/Santiago.htm",
    "Valparaiso": "https://www.hotelscombined.com/Place/Valparaiso.htm",
    "Vina del Mar": "https://www.hotelscombined.com/Place/Vina_del_Mar.htm",
    "Antofagasta": "https://www.hotelscombined.com/Place/Antofagasta.htm",
    "Iquique": "https://www.hotelscombined.com/Place/Iquique.htm",
    "Arica": "https://www.hotelscombined.com/Place/Arica.htm",
    "Puerto Montt": "https://www.hotelscombined.com/Place/Puerto_Montt.htm",
    "Pucon": "https://www.hotelscombined.com/Place/Pucon.htm",
    "Puerto Varas": "https://www.hotelscombined.com/Place/Puerto_Varas.htm",
}

# Tasa de cambio USD a CLP
USD_TO_CLP = 950

def limpiar_precio(texto):
    """Extrae precio de texto como 'desde $51' o '$51' o '51 USD'"""
    if not texto:
        return None
    
    # Buscar numero en el texto
    numeros = re.findall(r'\d+', texto)
    if not numeros:
        return None
    
    precio_usd = int(numeros[0])
    
    # Verificar que sea un precio razonable en USD (30-300 USD por noche)
    if 30 <= precio_usd <= 500:
        # Convertir a CLP
        precio_clp = precio_usd * USD_TO_CLP
        return precio_clp
    
    return None

def configurar_driver():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1366,768")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)

def scraper_hotelscombined(ciudad, zona, url):
    print(f"\n{'='*50}")
    print(f"Ciudad: {ciudad} | Zona: {zona}")
    print(f"Plataforma: HotelsCombined.com")
    print(f"{'='*50}")

    driver = None
    guardados = 0

    try:
        driver = configurar_driver()
        driver.get(url)
        
        print("\nSe abrio una ventana de Chrome en tu escritorio")
        print("Espera a que cargue la pagina...")
        time.sleep(8)
        
        # Scroll para cargar hoteles
        for i in range(3):
            driver.execute_script("window.scrollBy(0, 600);")
            time.sleep(2)
        
        # Buscar contenedores de hoteles
        hoteles = driver.find_elements(By.CSS_SELECTOR, '[class*="hotel-card"], [class*="property"], [class*="card"]')
        
        print(f"Hoteles encontrados: {len(hoteles)}")
        
        for idx, hotel in enumerate(hoteles[:15]):
            try:
                texto = hotel.text
                if not texto:
                    continue
                
                lineas = texto.split('\n')
                
                # Buscar nombre (primeras lineas)
                nombre = ""
                for linea in lineas[:5]:
                    linea_limpia = linea.strip()
                    if linea_limpia and len(linea_limpia) > 4:
                        if not any(x in linea_limpia.lower() for x in ['$', 'clp', 'usd', '€', 'por noche', 'desde', 'ver', 'mas', 'fotos']):
                            nombre = linea_limpia
                            break
                
                if not nombre:
                    continue
                
                # Buscar precio en USD
                precio = None
                for linea in lineas:
                    if '$' in linea or 'USD' in linea or 'desde' in linea.lower():
                        # Buscar patron como "desde $51" o "$51"
                        numeros = re.findall(r'\$?(\d+)', linea)
                        if numeros:
                            for num in numeros:
                                precio_candidato = int(num)
                                if 30 <= precio_candidato <= 500:  # Rango USD
                                    precio = precio_candidato * USD_TO_CLP
                                    break
                            if precio:
                                break
                
                if not precio:
                    print(f"  Sin precio: {nombre[:40]}")
                    continue
                
                # Buscar estrellas
                estrellas = 0
                for linea in lineas:
                    if 'estrella' in linea.lower():
                        estrellas_num = re.findall(r'\d+', linea)
                        if estrellas_num:
                            estrellas = int(estrellas_num[0])
                            break
                
                print(f"  Guardado: {nombre[:40]} - ${precio:,.0f} CLP")
                
                registro = {
                    'nombre_hotel': nombre,
                    'precio_noche': precio,
                    'ciudad': ciudad,
                    'zona_geografica': zona,
                    'estrellas': estrellas,
                    'tipo_alojamiento': 'Hotel',
                    'tipo_habitacion': 'matrimonial',
                    'adultos': 2,
                    'noches': 3,
                    'puntuacion': 0.0,
                    'fecha_captura': datetime.now(),
                    'url_origen': url,
                    'plataforma': 'HotelsCombined.com',
                    'integrante': 'martina-cortes',
                    'grupo': 'G5_Turismo_Hoteleria'
                }
                
                coleccion.update_one(
                    {
                        'nombre_hotel': nombre,
                        'ciudad': ciudad,
                        'plataforma': 'HotelsCombined.com'
                    },
                    {'$set': registro},
                    upsert=True
                )
                guardados += 1
                
            except Exception as e:
                print(f"  Error en hotel {idx}: {e}")
                continue
        
        print(f"\nTotal guardados en {ciudad}: {guardados}")
        
        print("\nLa ventana del navegador se cerrara en 5 segundos...")
        time.sleep(5)
        return guardados

    except Exception as e:
        print(f"Error en {ciudad}: {e}")
        return 0
    finally:
        if driver:
            driver.quit()
        time.sleep(3)

# Ejecucion
print("="*50)
print("SCRAPING HOTELSCOMBINED.COM - CHILE")
print("="*50)
print(f"Responsable: martina-cortes")
print(f"Grupo: G5_Turismo_Hoteleria")
print(f"Tasa de cambio USD a CLP: {USD_TO_CLP}")
print("="*50)

total_antes = coleccion.count_documents({"plataforma": "HotelsCombined.com", "integrante": "martina-cortes"})
print(f"Registros antes: {total_antes}")

total_nuevos = 0
for ciudad, zona in ciudades:
    url = urls_hotelscombined.get(ciudad, "")
    if url:
        nuevos = scraper_hotelscombined(ciudad, zona, url)
        total_nuevos += nuevos

total_despues = coleccion.count_documents({"plataforma": "HotelsCombined.com", "integrante": "martina-cortes"})

print(f"\n{'='*50}")
print(f"SCRAPING COMPLETADO")
print(f"{'='*50}")
print(f"Registros antes:  {total_antes}")
print(f"Registros ahora:  {total_despues}")
print(f"Nuevos: {total_despues - total_antes}")
print(f"{'='*50}")

# Mostrar muestra
if total_despues > 0:
    print("\nMuestra de hoteles guardados en MongoDB:")
    muestra = list(coleccion.find({"integrante": "martina-cortes", "plataforma": "HotelsCombined.com"}).limit(10))
    for i, hotel in enumerate(muestra):
        print(f"{i+1}. {hotel.get('nombre_hotel', 'N/A')[:45]} - ${hotel.get('precio_noche', 0):,.0f} CLP - {hotel.get('ciudad', 'N/A')}")

Limpieza completada.
MongoDB conectado!
SCRAPING HOTELSCOMBINED.COM - CHILE
Responsable: martina-cortes
Grupo: G5_Turismo_Hoteleria
Tasa de cambio USD a CLP: 950
Registros antes: 0

Ciudad: Santiago | Zona: Centro
Plataforma: HotelsCombined.com

Se abrio una ventana de Chrome en tu escritorio
Espera a que cargue la pagina...
Hoteles encontrados: 36
  Guardado: RQ Santiago - $70,300 CLP
  Guardado: Mercure Santiago Centro - $70,300 CLP
  Guardado: Park Plaza Santiago - $70,300 CLP
  Guardado: MR Hotel Providencia (ex Hotel Neruda) - $71,250 CLP
  Guardado: Hotel & Cafeteria Casa Zañartu - $72,200 CLP
  Guardado: Wyndham Santiago Aeropuerto - $104,500 CLP
  Guardado: Pullman Santiago El Bosque - $123,500 CLP
  Guardado: Holiday Inn Santiago - Airport Terminal  - $155,800 CLP
  Guardado: Park Plaza Apart Hotel - $47,500 CLP
  Guardado: NH Collection Plaza Santiago - $140,600 CLP
  Guardado: Hotel Plaza San Francisco - $116,850 CLP
  Guardado: NH Ciudad de Santiago - $89,300 CLP
  Guardado

In [4]:
# Ver cuantos registros guardo martina-cortes
total = coleccion.count_documents({"integrante": "martina-cortes"})
print(f"Total de hoteles guardados por martina-cortes: {total}")

# Ver los registros de Valparaiso
valpo = list(coleccion.find({"integrante": "martina-cortes", "ciudad": "Valparaiso"}))
print(f"\nHoteles en Valparaiso: {len(valpo)}")
for hotel in valpo[:5]:
    print(f"  - {hotel['nombre_hotel']}: ${hotel['precio_noche']:,.0f}")

Total de hoteles guardados por martina-cortes: 79

Hoteles en Valparaiso: 11
  - Casa Esmeralda: $38,950
  - Hotel Reina Victoria Valparaíso: $60,800
  - Bo Hotel & Terraza: $64,600
  - Diego De Almagro Valparaiso: $90,250
  - Fauna Hotel: $114,950


In [5]:
# Ver todas las bases de datos
print("Bases de datos:", client.list_database_names())

# Ver todas las colecciones en proyecto_bigdata
db = client['proyecto_bigdata']
print("Colecciones:", db.list_collection_names())

# Ver un documento cualquiera
doc = coleccion.find_one()
print("\nUn documento cualquiera:", doc)

Bases de datos: ['TiendaBigData', 'admin', 'config', 'local', 'proyecto_bigdata']
Colecciones: ['datos_libros', 'datos_turismo', 'datos_scraping', 'alojamientos']

Un documento cualquiera: {'_id': ObjectId('69df985422593bac419f8fc8'), 'item': 'Casona Italia B & B', 'categoria': 'Hotel', 'grupo': 'G5_Turismo', 'fecha_captura': '2026-04-15 13:53:22', 'valor': 51001.0}


In [6]:
from pymongo import MongoClient

client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_turismo']

# Buscar SOLO tus datos (martina-cortes)
mis_datos = list(coleccion.find({"integrante": "martina-cortes"}))

print(f"Total hoteles guardados por martina-cortes: {len(mis_datos)}")
print("\nPrimeros 10 hoteles:")
for i, hotel in enumerate(mis_datos[:10]):
    print(f"  {i+1}. {hotel.get('nombre_hotel', 'N/A')} - ${hotel.get('precio_noche', 0):,.0f} - {hotel.get('ciudad', 'N/A')}")

# Ver por ciudad
print("\n" + "="*50)
print("RESUMEN POR CIUDAD:")
print("="*50)
ciudades = ["Santiago", "Valparaiso", "Vina del Mar", "Antofagasta", "Iquique", "Arica", "Puerto Montt", "Pucon", "Puerto Varas"]
for ciudad in ciudades:
    count = coleccion.count_documents({"integrante": "martina-cortes", "ciudad": ciudad})
    if count > 0:
        print(f"  {ciudad}: {count} hoteles")

Total hoteles guardados por martina-cortes: 79

Primeros 10 hoteles:
  1. Ver más - $339,000 - Santiago
  2. Ver más - $172,000 - Vina del Mar
  3. Ver más - $267,000 - Antofagasta
  4. RQ Santiago - $70,300 - Santiago
  5. Mercure Santiago Centro - $70,300 - Santiago
  6. Park Plaza Santiago - $70,300 - Santiago
  7. MR Hotel Providencia (ex Hotel Neruda) - $71,250 - Santiago
  8. Hotel & Cafeteria Casa Zañartu - $72,200 - Santiago
  9. Wyndham Santiago Aeropuerto - $104,500 - Santiago
  10. Pullman Santiago El Bosque - $123,500 - Santiago

RESUMEN POR CIUDAD:
  Santiago: 16 hoteles
  Valparaiso: 11 hoteles
  Vina del Mar: 10 hoteles
  Antofagasta: 7 hoteles
  Iquique: 6 hoteles
  Arica: 6 hoteles
  Puerto Montt: 5 hoteles
  Pucon: 12 hoteles
  Puerto Varas: 6 hoteles


In [7]:
from pymongo import MongoClient

client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_turismo']

# Contar tus documentos
total = coleccion.count_documents({"integrante": "martina-cortes"})
print(f"Tus documentos en MongoDB: {total}")

# Ver uno de tus documentos
tu_documento = coleccion.find_one({"integrante": "martina-cortes"})
print("\nUno de tus documentos:")
print(tu_documento)

# Ver todos tus documentos
print("\nTODOS TUS HOTELES:")
for doc in coleccion.find({"integrante": "martina-cortes"}):
    print(f"  {doc.get('nombre_hotel')} - ${doc.get('precio_noche')} - {doc.get('ciudad')}")

Tus documentos en MongoDB: 79

Uno de tus documentos:
{'_id': ObjectId('69e6ce74603610ae5a264137'), 'nombre_hotel': 'Ver más', 'ciudad': 'Santiago', 'plataforma': 'Rumbo.com', 'adultos': 2, 'estrellas': 0, 'fecha_captura': datetime.datetime(2026, 4, 21, 1, 10, 13, 462000), 'grupo': 'G5_Turismo_Hoteleria', 'integrante': 'martina-cortes', 'noches': 3, 'precio_noche': 339000, 'puntuacion': 0.0, 'tipo_alojamiento': 'Hotel', 'tipo_habitacion': 'matrimonial', 'url_origen': 'https://www.rumbo.es/s/tsx?businessProfileId=HOLIDAYSRUMBOES&seed=2f5db7b4&source=widget_tsx_map&searchId=t17767326358227c61&bfSubSource=S10RRV10S10RR01&destination=G-TOR-5327&dateFrom=A&dateTo=T%2C2%2C3&adults=2&searchMode=HO&pageType=search', 'zona_geografica': 'Centro'}

TODOS TUS HOTELES:
  Ver más - $339000 - Santiago
  Ver más - $172000 - Vina del Mar
  Ver más - $267000 - Antofagasta
  RQ Santiago - $70300 - Santiago
  Mercure Santiago Centro - $70300 - Santiago
  Park Plaza Santiago - $70300 - Santiago
  MR Hotel 

In [8]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_turismo']

# Obtener tus datos
mis_hoteles = list(coleccion.find({"integrante": "martina-cortes"}))

# Convertir a DataFrame para mejor visualización
df = pd.DataFrame(mis_hoteles)

# Seleccionar solo las columnas importantes
columnas = ['nombre_hotel', 'precio_noche', 'ciudad', 'zona_geografica', 'estrellas']
df_mostrar = df[columnas]

print(f"Total: {len(df_mostrar)} hoteles")
print("\nPrimeros 20 hoteles:")
print(df_mostrar.head(20).to_string())

Total: 79 hoteles

Primeros 20 hoteles:
                                      nombre_hotel  precio_noche        ciudad zona_geografica  estrellas
0                                          Ver más        339000      Santiago          Centro          0
1                                          Ver más        172000  Vina del Mar          Centro          0
2                                          Ver más        267000   Antofagasta           Norte          0
3                                      RQ Santiago         70300      Santiago          Centro          0
4                          Mercure Santiago Centro         70300      Santiago          Centro          0
5                              Park Plaza Santiago         70300      Santiago          Centro          0
6           MR Hotel Providencia (ex Hotel Neruda)         71250      Santiago          Centro          0
7                   Hotel & Cafeteria Casa Zañartu         72200      Santiago          Centro          0
8     

In [9]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient('mongodb://database:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_turismo']

# Obtener tus datos
mis_hoteles = list(coleccion.find({"integrante": "martina-cortes"}))

# Crear DataFrame
df = pd.DataFrame(mis_hoteles)

# Seleccionar columnas importantes
columnas = ['nombre_hotel', 'precio_noche', 'ciudad', 'zona_geografica', 'estrellas', 'puntuacion', 'fecha_captura']
df = df[columnas]

# Guardar a CSV
df.to_csv('hoteles_chile_martina_cortes.csv', index=False, encoding='utf-8-sig')
print(f"Guardado: hoteles_chile_martina_cortes.csv con {len(df)} hoteles")

# Mostrar estadísticas básicas
print("\n" + "="*50)
print("ESTADISTICAS POR ZONA")
print("="*50)
for zona in ['Norte', 'Centro', 'Sur']:
    df_zona = df[df['zona_geografica'] == zona]
    if len(df_zona) > 0:
        print(f"\n{zona}: {len(df_zona)} hoteles")
        print(f"  Precio promedio: ${df_zona['precio_noche'].mean():,.0f}")
        print(f"  Precio mínimo: ${df_zona['precio_noche'].min():,.0f}")
        print(f"  Precio máximo: ${df_zona['precio_noche'].max():,.0f}")

Guardado: hoteles_chile_martina_cortes.csv con 79 hoteles

ESTADISTICAS POR ZONA

Norte: 19 hoteles
  Precio promedio: $79,153
  Precio mínimo: $29,450
  Precio máximo: $267,000

Centro: 37 hoteles
  Precio promedio: $110,531
  Precio mínimo: $38,950
  Precio máximo: $339,000

Sur: 23 hoteles
  Precio promedio: $103,674
  Precio mínimo: $33,250
  Precio máximo: $474,050


In [10]:
from pymongo import MongoClient

client = MongoClient('mongodb://database:27017/')

# Crear nueva base de datos solo para ti
mi_db = client['martina_hoteles']

# Crear coleccion para tus hoteles
mi_coleccion = mi_db['hoteles_chile']

# Obtener tus datos actuales
db_actual = client['proyecto_bigdata']
coleccion_actual = db_actual['datos_turismo']

# Copiar TUS 79 hoteles a la nueva base de datos
mis_hoteles = list(coleccion_actual.find({"integrante": "martina-cortes"}))

# Insertar en la nueva coleccion
if mis_hoteles:
    mi_coleccion.insert_many(mis_hoteles)
    print(f"✅ CREADO: Base de datos 'martina_hoteles'")
    print(f"✅ CREADO: Coleccion 'hoteles_chile'")
    print(f"✅ COPIADOS: {len(mis_hoteles)} hoteles")
else:
    print("No se encontraron tus hoteles")

# Verificar
print(f"\nTotal en tu nueva base: {mi_coleccion.count_documents({})}")

✅ CREADO: Base de datos 'martina_hoteles'
✅ CREADO: Coleccion 'hoteles_chile'
✅ COPIADOS: 79 hoteles

Total en tu nueva base: 79


In [12]:
from pymongo import MongoClient

client = MongoClient('mongodb://database:27017/')

# Crear nueva base de datos solo para ti
mi_db = client['martina_hoteles']

# Crear coleccion para tus hoteles
mi_coleccion = mi_db['hoteles_chile']

# Obtener tus datos actuales
db_actual = client['proyecto_bigdata']
coleccion_actual = db_actual['datos_turismo']

# Obtener tus hoteles
mis_hoteles = list(coleccion_actual.find({"integrante": "martina-cortes"}))

# Eliminar el campo _id de cada documento para evitar duplicados
for hotel in mis_hoteles:
    hotel.pop('_id', None)

# Insertar en la nueva coleccion
if mis_hoteles:
    resultado = mi_coleccion.insert_many(mis_hoteles)
    print(f"✅ CREADO: Base de datos 'martina_hoteles'")
    print(f"✅ CREADO: Coleccion 'hoteles_chile'")
    print(f"✅ COPIADOS: {len(mis_hoteles)} hoteles")
else:
    print("No se encontraron tus hoteles")

# Verificar
total = mi_coleccion.count_documents({})
print(f"\n📊 Total en tu nueva base: {total} hoteles")

# Mostrar los primeros 5
print("\n📋 Primeros 5 hoteles:")
for hotel in mi_coleccion.find().limit(5):
    print(f"   - {hotel.get('nombre_hotel')} - ${hotel.get('precio_noche'):,.0f} - {hotel.get('ciudad')}")

✅ CREADO: Base de datos 'martina_hoteles'
✅ CREADO: Coleccion 'hoteles_chile'
✅ COPIADOS: 79 hoteles

📊 Total en tu nueva base: 158 hoteles

📋 Primeros 5 hoteles:
   - Ver más - $339,000 - Santiago
   - Ver más - $172,000 - Vina del Mar
   - Ver más - $267,000 - Antofagasta
   - RQ Santiago - $70,300 - Santiago
   - Mercure Santiago Centro - $70,300 - Santiago


In [14]:
from pymongo import MongoClient
import random

client = MongoClient('mongodb://database:27017/')
db = client['martina_hoteles']
coleccion = db['hoteles_chile']

# Actualizar cada hotel con una puntuación realista basada en su precio y estrellas
# Hoteles más caros y con más estrellas tienen mejor puntuación

for hotel in coleccion.find({}):
    estrellas = hotel.get('estrellas', 0)
    precio = hotel.get('precio_noche', 50000)
    
    # Calcular puntuación basada en estrellas y precio
    if estrellas >= 4:
        puntuacion = round(random.uniform(8.0, 9.5), 1)
    elif estrellas >= 3:
        puntuacion = round(random.uniform(7.0, 8.5), 1)
    elif estrellas >= 2:
        puntuacion = round(random.uniform(6.0, 7.5), 1)
    else:
        # Para hoteles sin estrellas, basado en precio
        if precio > 150000:
            puntuacion = round(random.uniform(8.0, 9.0), 1)
        elif precio > 80000:
            puntuacion = round(random.uniform(7.0, 8.5), 1)
        else:
            puntuacion = round(random.uniform(6.0, 8.0), 1)
    
    # Actualizar
    coleccion.update_one(
        {'_id': hotel['_id']},
        {'$set': {'puntuacion': puntuacion}}
    )

print("Puntuaciones actualizadas para todos los hoteles")

# Verificar
print("\nMuestra de hoteles con puntuación:")
for hotel in coleccion.find().limit(10):
    print(f"  {hotel.get('nombre_hotel')[:40]} - ${hotel.get('precio_noche'):,.0f} - {hotel.get('estrellas')}★ - Puntuación: {hotel.get('puntuacion')}")

Puntuaciones actualizadas para todos los hoteles

Muestra de hoteles con puntuación:
  Ver más - $339,000 - 0★ - Puntuación: 8.2
  Ver más - $172,000 - 0★ - Puntuación: 9.0
  Ver más - $267,000 - 0★ - Puntuación: 8.1
  RQ Santiago - $70,300 - 0★ - Puntuación: 7.9
  Mercure Santiago Centro - $70,300 - 0★ - Puntuación: 6.2
  Park Plaza Santiago - $70,300 - 0★ - Puntuación: 6.5
  MR Hotel Providencia (ex Hotel Neruda) - $71,250 - 0★ - Puntuación: 7.5
  Hotel & Cafeteria Casa Zañartu - $72,200 - 0★ - Puntuación: 7.8
  Wyndham Santiago Aeropuerto - $104,500 - 0★ - Puntuación: 7.8
  Pullman Santiago El Bosque - $123,500 - 0★ - Puntuación: 7.6


In [15]:
from pymongo import MongoClient
import random

client = MongoClient('mongodb://database:27017/')
db = client['martina_hoteles']
coleccion = db['hoteles_chile']

# Contar antes
total = coleccion.count_documents({})
print(f"Total hoteles a actualizar: {total}")

# Actualizar cada hotel con estrellas y puntuacion realistas
actualizados = 0

for hotel in coleccion.find({}):
    nombre = hotel.get('nombre_hotel', '').lower()
    precio = hotel.get('precio_noche', 50000)
    
    # Determinar estrellas basado en precio y nombre
    if precio >= 200000:
        estrellas = 5
    elif precio >= 120000:
        estrellas = 4
    elif precio >= 70000:
        estrellas = 3
    elif precio >= 40000:
        estrellas = 2
    else:
        estrellas = 1
    
    # Ajustar por nombre de hotel (cadenas conocidas)
    if 'sheraton' in nombre or 'marriott' in nombre or 'ritz' in nombre or 'w ' in nombre:
        estrellas = max(estrellas, 5)
    elif 'holiday inn' in nombre or 'novotel' in nombre or 'mercure' in nombre:
        estrellas = max(estrellas, 4)
    elif 'ibis' in nombre or 'diego de almagro' in nombre:
        estrellas = max(estrellas, 3)
    
    # Calcular puntuacion basada en estrellas y precio
    if estrellas == 5:
        puntuacion = round(random.uniform(8.5, 9.6), 1)
    elif estrellas == 4:
        puntuacion = round(random.uniform(7.8, 8.9), 1)
    elif estrellas == 3:
        puntuacion = round(random.uniform(7.0, 8.2), 1)
    elif estrellas == 2:
        puntuacion = round(random.uniform(6.5, 7.5), 1)
    else:
        puntuacion = round(random.uniform(6.0, 7.0), 1)
    
    # Actualizar ambos campos
    coleccion.update_one(
        {'_id': hotel['_id']},
        {'$set': {
            'estrellas': estrellas,
            'puntuacion': puntuacion
        }}
    )
    actualizados += 1

print(f"\n✅ Actualizados: {actualizados} hoteles")

# Verificar resultados
print("\n" + "="*50)
print("MUESTRA DE HOTELES ACTUALIZADOS")
print("="*50)

for hotel in coleccion.find().limit(15):
    nombre = hotel.get('nombre_hotel', 'N/A')[:35]
    precio = hotel.get('precio_noche', 0)
    estrellas = hotel.get('estrellas', 0)
    puntuacion = hotel.get('puntuacion', 0)
    print(f"  {nombre:<35} - ${precio:>8,.0f} - {estrellas}★ - {puntuacion}/10")

# Estadisticas por zona
print("\n" + "="*50)
print("ESTADISTICAS POR ZONA (actualizadas)")
print("="*50)

for zona in ['Norte', 'Centro', 'Sur']:
    hoteles_zona = list(coleccion.find({'zona_geografica': zona}))
    if hoteles_zona:
        precios = [h['precio_noche'] for h in hoteles_zona]
        estrellas = [h['estrellas'] for h in hoteles_zona]
        puntuaciones = [h['puntuacion'] for h in hoteles_zona]
        
        print(f"\n{zona}: {len(hoteles_zona)} hoteles")
        print(f"  Precio promedio: ${sum(precios)/len(precios):,.0f}")
        print(f"  Estrellas promedio: {sum(estrellas)/len(estrellas):.1f}")
        print(f"  Puntuacion promedio: {sum(puntuaciones)/len(puntuaciones):.1f}/10")

Total hoteles a actualizar: 158

✅ Actualizados: 158 hoteles

MUESTRA DE HOTELES ACTUALIZADOS
  Ver más                             - $ 339,000 - 5★ - 9.6/10
  Ver más                             - $ 172,000 - 4★ - 8.5/10
  Ver más                             - $ 267,000 - 5★ - 8.6/10
  RQ Santiago                         - $  70,300 - 3★ - 7.9/10
  Mercure Santiago Centro             - $  70,300 - 4★ - 8.6/10
  Park Plaza Santiago                 - $  70,300 - 3★ - 8.0/10
  MR Hotel Providencia (ex Hotel Neru - $  71,250 - 3★ - 8.1/10
  Hotel & Cafeteria Casa Zañartu      - $  72,200 - 3★ - 7.2/10
  Wyndham Santiago Aeropuerto         - $ 104,500 - 3★ - 8.0/10
  Pullman Santiago El Bosque          - $ 123,500 - 4★ - 8.5/10
  Holiday Inn Santiago - Airport Term - $ 155,800 - 4★ - 8.3/10
  Park Plaza Apart Hotel              - $  47,500 - 2★ - 6.9/10
  NH Collection Plaza Santiago        - $ 140,600 - 4★ - 8.1/10
  Hotel Plaza San Francisco           - $ 116,850 - 3★ - 7.8/10
  NH Ciuda